# 11 — Chroma Metric Comparison

Build three Chroma collections (`cosine`, `l2`, `ip`) and compare.

| Collection | `hnsw:space` | Notes |
|---|---|---|
| `arxiv_cosine` | `cosine` | Recommended for normalized embeddings |
| `arxiv_l2` | `l2` | Default; raw Euclidean |
| `arxiv_ip` | `ip` | Inner product |

**Config source:** `configs/default.yaml` → `paths.chroma_experiment`

In [ ]:
from rag_pipeline.utils import load_notebook_config
from rag_pipeline.embeddings import build_embeddings
from rag_pipeline.ingestion import load_documents
from rag_pipeline.splitting import split_documents
from rag_pipeline.vectorstores import build_vectorstore

cfg, REPO = load_notebook_config()
QUERIES = cfg.notebooks["retrieval_queries"]
CHROMA_DIR = REPO / cfg.paths["chroma_experiment"]

**Load + chunk + embed**

In [ ]:
docs = load_documents(
    cfg.data["sources"],
    sample_fraction=cfg.data.get("sample_fraction"),
    sample_size=cfg.data.get("sample_size"),
    sample_seed=cfg.data.get("sample_seed", 42),
)[:500]
chunks = split_documents(docs, dict(cfg.splitting))
emb = build_embeddings(dict(cfg.embeddings))
print(f"Chunks: {len(chunks)} | Chroma dir: {CHROMA_DIR}")

**Build collections**

In [ ]:
METRICS = ["cosine", "l2", "ip"]

stores = {
    metric: build_vectorstore(chunks, emb, {
        "type": "chroma",
        "persist_dir": str(CHROMA_DIR),
        "collection_name": f"arxiv_{metric}",
        "metric": metric,
    })
    for metric in METRICS
}

for metric, store in stores.items():
    print(f"{metric:<8} count = {store._collection.count()}")

**Query all three**

In [ ]:
for q in QUERIES[:3]:
    print(f"\n{'='*80}\nQuery: {q}\n{'='*80}")
    for metric, store in stores.items():
        print(f"\n{metric.upper()}")
        print("-" * 78)
        for rank, (doc, score) in enumerate(store.similarity_search_with_score(q, k=5), 1):
            row = doc.metadata.get("row", "?")
            preview = doc.page_content.replace("\n", " ")[:70]
            print(f"  {rank}. row={row:>4}  score={score:>8.4f}  {preview}...")